# HOG Feature Visualization

This notebook demonstrates HOG (Histogram of Oriented Gradients) feature extraction and visualization.

In [ ]:
# Import necessary libraries
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from skimage.feature import hog
from skimage import exposure

# Add src directory to path
import sys
sys.path.append('../src')

# Import project modules
from config import TRAIN_DIR, HOG_ORIENTATIONS, HOG_PIXELS_PER_CELL, HOG_CELLS_PER_BLOCK, HOG_BLOCK_NORM
from preprocess import load_images_from_directory, preprocess_images
from features import extract_hog_features, visualize_hog

## Load Sample Images

In [ ]:
# Load sample images
train_images, train_labels = load_images_from_directory(TRAIN_DIR)

# Preprocess images
processed_images = preprocess_images(train_images[:5])  # Take first 5 images

print(f"Loaded {len(processed_images)} sample images for HOG visualization")
print(f"Image shape: {processed_images[0].shape}")
print(f"HOG parameters:")
print(f"  Orientations: {HOG_ORIENTATIONS}")
print(f"  Pixels per cell: {HOG_PIXELS_PER_CELL}")
print(f"  Cells per block: {HOG_CELLS_PER_BLOCK}")
print(f"  Block normalization: {HOG_BLOCK_NORM}")

## HOG Feature Extraction and Visualization

In [ ]:
# Extract and visualize HOG features for sample images
fig, axes = plt.subplots(len(processed_images), 3, figsize=(15, 5*len(processed_images)))

for i, img in enumerate(processed_images):
    # Convert to uint8 for display
    img_display = (img * 255).astype(np.uint8)
    
    # Extract HOG features
    hog_features = extract_hog_features(img)
    
    # Get HOG visualization
    hog_vis = visualize_hog(img_display)
    
    # Convert to grayscale for HOG
    gray = cv2.cvtColor(img_display, cv2.COLOR_RGB2GRAY)
    
    # Plot original image
    axes[i, 0].imshow(img_display)
    axes[i, 0].set_title(f'Original - Class: {train_labels[i]}')
    axes[i, 0].axis('off')
    
    # Plot grayscale
    axes[i, 1].imshow(gray, cmap='gray')
    axes[i, 1].set_title('Grayscale')
    axes[i, 1].axis('off')
    
    # Plot HOG visualization
    axes[i, 2].imshow(hog_vis, cmap='gray')
    axes[i, 2].set_title(f'HOG Features (Size: {len(hog_features)})')
    axes[i, 2].axis('off')

plt.tight_layout()
plt.show()

## HOG Parameter Analysis

In [ ]:
# Test different HOG parameters
sample_img = processed_images[0]
img_display = (sample_img * 255).astype(np.uint8)
gray = cv2.cvtColor(img_display, cv2.COLOR_RGB2GRAY)

# Different parameter combinations
param_sets = [
    {'orientations': 9, 'pixels_per_cell': (8, 8), 'cells_per_block': (2, 2)},
    {'orientations': 6, 'pixels_per_cell': (8, 8), 'cells_per_block': (2, 2)},
    {'orientations': 12, 'pixels_per_cell': (8, 8), 'cells_per_block': (2, 2)},
    {'orientations': 9, 'pixels_per_cell': (4, 4), 'cells_per_block': (2, 2)},
    {'orientations': 9, 'pixels_per_cell': (16, 16), 'cells_per_block': (2, 2)},
    {'orientations': 9, 'pixels_per_cell': (8, 8), 'cells_per_block': (1, 1)},
    {'orientations': 9, 'pixels_per_cell': (8, 8), 'cells_per_block': (3, 3)}
]

fig, axes = plt.subplots(3, 3, figsize=(15, 12))
axes = axes.flatten()

# Original image
axes[0].imshow(gray, cmap='gray')
axes[0].set_title('Original Grayscale')
axes[0].axis('off')

# Different HOG configurations
for i, params in enumerate(param_sets[:8]):
    hog_features, hog_image = hog(
        gray,
        orientations=params['orientations'],
        pixels_per_cell=params['pixels_per_cell'],
        cells_per_block=params['cells_per_block'],
        block_norm='L2-Hys',
        visualize=True,
        feature_vector=True
    )
    
    # Enhance contrast for better visualization
    hog_image_rescaled = exposure.rescale_intensity(hog_image, in_range=(0, 10))
    
    axes[i+1].imshow(hog_image_rescaled, cmap='gray')
    axes[i+1].set_title(f"O:{params['orientations']}, PC:{params['pixels_per_cell']}, CB:{params['cells_per_block']}\nFeatures: {len(hog_features)}")
    axes[i+1].axis('off')

plt.tight_layout()
plt.show()

## Feature Vector Analysis

In [ ]:
# Analyze HOG feature vectors
from features import extract_hog_features_batch

# Extract HOG features for all sample images
hog_features_batch = extract_hog_features_batch(processed_images)

print(f"HOG feature vectors shape: {hog_features_batch.shape}")
print(f"Feature vector length: {hog_features_batch.shape[1]}")

# Feature statistics
feature_means = np.mean(hog_features_batch, axis=0)
feature_stds = np.std(hog_features_batch, axis=0)
feature_mins = np.min(hog_features_batch, axis=0)
feature_maxs = np.max(hog_features_batch, axis=0)

print(f"\nFeature Statistics:")
print(f"  Mean: {np.mean(feature_means):.4f}")
print(f"  Std: {np.mean(feature_stds):.4f}")
print(f"  Min: {np.min(feature_mins):.4f}")
print(f"  Max: {np.max(feature_maxs):.4f}")

# Plot feature distribution
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Histogram of all feature values
all_features = hog_features_batch.flatten()
ax1.hist(all_features, bins=50, alpha=0.7, color='skyblue', density=True)
ax1.set_title('Distribution of All HOG Feature Values')
ax1.set_xlabel('Feature Value')
ax1.set_ylabel('Density')

# Feature means per image
ax2.bar(range(len(processed_images)), np.mean(hog_features_batch, axis=1), color='lightcoral')
ax2.set_title('Mean Feature Value per Image')
ax2.set_xlabel('Image Index')
ax2.set_ylabel('Mean Feature Value')

# Add class labels
for i, label in enumerate(train_labels[:len(processed_images)]):
    ax2.text(i, np.mean(hog_features_batch, axis=1)[i], label, 
             ha='center', va='bottom')

plt.tight_layout()
plt.show()

## Class-wise Feature Analysis

In [ ]:
# Analyze features by class
unique_labels = list(set(train_labels))
class_features = {}

for label in unique_labels:
    # Get indices of images for this class
    class_indices = [i for i, l in enumerate(train_labels) if l == label]
    
    # Extract features for this class (sample up to 10 images)
    class_images = [processed_images[i] for i in class_indices[:10]]
    class_hog_features = extract_hog_features_batch(class_images)
    
    class_features[label] = class_hog_features
    
    print(f"Class {label}: {len(class_images)} images, features shape: {class_hog_features.shape}")

# Compare feature distributions between classes
if len(unique_labels) >= 2:
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    # Plot mean feature vectors
    for i, label in enumerate(unique_labels):
        features = class_features[label]
        mean_features = np.mean(features, axis=0)
        
        axes[0, 0].plot(mean_features, label=f'Class {label}', alpha=0.7)
        axes[0, 1].hist(mean_features, bins=30, alpha=0.5, label=f'Class {label}')
    
    axes[0, 0].set_title('Mean HOG Feature Vectors by Class')
    axes[0, 0].set_xlabel('Feature Index')
    axes[0, 0].set_ylabel('Mean Feature Value')
    axes[0, 0].legend()
    
    axes[0, 1].set_title('Distribution of Mean Feature Values')
    axes[0, 1].set_xlabel('Mean Feature Value')
    axes[0, 1].set_ylabel('Frequency')
    axes[0, 1].legend()
    
    # Plot feature variance
    for i, label in enumerate(unique_labels):
        features = class_features[label]
        var_features = np.var(features, axis=0)
        
        axes[1, 0].plot(var_features, label=f'Class {label}', alpha=0.7)
        axes[1, 1].hist(var_features, bins=30, alpha=0.5, label=f'Class {label}')
    
    axes[1, 0].set_title('Feature Variance by Class')
    axes[1, 0].set_xlabel('Feature Index')
    axes[1, 0].set_ylabel('Feature Variance')
    axes[1, 0].legend()
    
    axes[1, 1].set_title('Distribution of Feature Variance')
    axes[1, 1].set_xlabel('Feature Variance')
    axes[1, 1].set_ylabel('Frequency')
    axes[1, 1].legend()
    
    plt.tight_layout()
    plt.show()
else:
    print("Need at least 2 classes for comparison")

## HOG Feature Importance Analysis

In [ ]:
# Analyze which HOG features are most discriminative
if len(unique_labels) >= 2:
    # Calculate feature importance using variance between classes
    class_means = {}
    for label in unique_labels:
        features = class_features[label]
        class_means[label] = np.mean(features, axis=0)
    
    # Calculate between-class variance
    feature_importance = np.var(list(class_means.values()), axis=0)
    
    # Sort features by importance
    important_indices = np.argsort(feature_importance)[::-1][:20]  # Top 20
    
    print(f"Top 10 most discriminative HOG features:")
    for i, idx in enumerate(important_indices[:10]):
        print(f"  Feature {idx}: Importance = {feature_importance[idx]:.6f}")
    
    # Visualize feature importance
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
    
    # Plot feature importance
    ax1.bar(range(20), feature_importance[important_indices[:20]], color='skyblue')
    ax1.set_title('Top 20 Most Discriminative HOG Features')
    ax1.set_xlabel('Feature Rank')
    ax1.set_ylabel('Feature Importance (Between-class variance)')
    
    # Plot cumulative importance
    sorted_importance = np.sort(feature_importance)[::-1]
    cumulative_importance = np.cumsum(sorted_importance) / np.sum(sorted_importance)
    
    ax2.plot(range(len(cumulative_importance)), cumulative_importance, color='lightcoral')
    ax2.set_title('Cumulative Feature Importance')
    ax2.set_xlabel('Number of Features')
    ax2.set_ylabel('Cumulative Importance')
    ax2.axhline(y=0.9, color='red', linestyle='--', alpha=0.7, label='90% importance')
    ax2.axhline(y=0.95, color='orange', linestyle='--', alpha=0.7, label='95% importance')
    ax2.legend()
    
    # Find number of features needed for 90% and 95% importance
    n_features_90 = np.where(cumulative_importance >= 0.9)[0][0] + 1
    n_features_95 = np.where(cumulative_importance >= 0.95)[0][0] + 1
    
    print(f"\nFeature reduction analysis:")
    print(f"  Total features: {len(feature_importance)}")
    print(f"  Features for 90% importance: {n_features_90} ({n_features_90/len(feature_importance)*100:.1f}%)")
    print(f"  Features for 95% importance: {n_features_95} ({n_features_95/len(feature_importance)*100:.1f}%)")
    
    plt.tight_layout()
    plt.show()
else:
    print("Need at least 2 classes for feature importance analysis")

## HOG vs Other Features Comparison

In [ ]:
# Compare HOG with other feature extraction methods
from features import extract_color_histogram, extract_lbp_features

sample_img = processed_images[0]
img_display = (sample_img * 255).astype(np.uint8)

# Extract different types of features
hog_feat = extract_hog_features(sample_img)
color_hist_feat = extract_color_histogram(img_display, bins=64)
lbp_feat = extract_lbp_features(img_display)

print(f"Feature extraction comparison:")
print(f"  HOG features: {len(hog_feat)}")
print(f"  Color histogram features: {len(color_hist_feat)}")
print(f"  LBP features: {len(lbp_feat)}")

# Visualize different feature types
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Original image
axes[0, 0].imshow(img_display)
axes[0, 0].set_title('Original Image')
axes[0, 0].axis('off')

# HOG visualization
hog_vis = visualize_hog(img_display)
axes[0, 1].imshow(hog_vis, cmap='gray')
axes[0, 1].set_title(f'HOG Features ({len(hog_feat)} dims)')
axes[0, 1].axis('off')

# Color histogram visualization
axes[1, 0].plot(color_hist_feat[:256], 'r-', alpha=0.7, label='Red')
axes[1, 0].plot(color_hist_feat[256:512], 'g-', alpha=0.7, label='Green')
axes[1, 0].plot(color_hist_feat[512:], 'b-', alpha=0.7, label='Blue')
axes[1, 0].set_title(f'Color Histogram ({len(color_hist_feat)} dims)')
axes[1, 0].set_xlabel('Bin')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].legend()

# LBP visualization
gray = cv2.cvtColor(img_display, cv2.COLOR_RGB2GRAY)
from skimage.feature import local_binary_pattern
lbp_image = local_binary_pattern(gray, 8, 1, method='uniform')
axes[1, 1].imshow(lbp_image, cmap='gray')
axes[1, 1].set_title(f'LBP Pattern ({len(lbp_feat)} dims)')
axes[1, 1].axis('off')

plt.tight_layout()
plt.show()

## Summary and Recommendations

In [ ]:
# Summary of HOG analysis
print("HOG Feature Analysis Summary:")
print("=" * 40)
print(f"Feature vector length: {hog_features_batch.shape[1]}")
print(f"Number of orientations: {HOG_ORIENTATIONS}")
print(f"Pixels per cell: {HOG_PIXELS_PER_CELL}")
print(f"Cells per block: {HOG_CELLS_PER_BLOCK}")

if len(unique_labels) >= 2:
    print(f"\nFeature discriminability:")
    print(f"  Mean between-class variance: {np.mean(feature_importance):.6f}")
    print(f"  Max between-class variance: {np.max(feature_importance):.6f}")
    print(f"  Features for 90% importance: {n_features_90} ({n_features_90/len(feature_importance)*100:.1f}%)")

print("\nRecommendations:")
print("1. HOG features provide good discriminability between classes")
print("2. Consider feature selection to reduce dimensionality if needed")
print("3. Current parameters provide good balance between detail and computational efficiency")
print("4. HOG features are robust to illumination changes")
print("5. Combine with color features for potentially better performance")